# 🖊️ Bengali Character Classification (BanglaLekha-Isolated)
**T3.5 — 3-Layer CNN trained from scratch**

**Dataset:** BanglaLekha-Isolated — 84 character classes, ~166k images  
**Source:** https://data.mendeley.com/datasets/hf6sf8zrkc/2  
**Target Accuracy:** 97%+ in < 5 minutes on Colab T4

---
### Pipeline Overview
1. Install dependencies & mount Google Drive (Colab)
2. Download & extract BanglaLekha-Isolated dataset
3. Explore & visualize dataset
4. Build data loaders with augmentation
5. Define 3-layer CNN (~300k params)
6. Train with OneCycleLR scheduler
7. Evaluate — confusion matrix, per-class accuracy
8. Save model & export for Streamlit app

## 1. Environment Setup

In [ ]:
# Install required packages
!pip install -q torch torchvision matplotlib seaborn scikit-learn tqdm pillow

# For downloading from Mendeley (direct link or via gdown)
!pip install -q gdown requests

In [ ]:
import os
import random
import zipfile
import urllib.request
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from pathlib import Path
from PIL import Image
from collections import Counter

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, random_split, Dataset
from torchvision import datasets, transforms
from torchvision.utils import make_grid
from sklearn.metrics import confusion_matrix, classification_report
from tqdm import tqdm

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.backends.cudnn.deterministic = True

# Device
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"✅ Using device: {DEVICE}")
if torch.cuda.is_available():
    print(f"   GPU: {torch.cuda.get_device_name(0)}")

## 2. Dataset Download & Extraction

> **Manual Download Option:**  
> If the automatic download fails, go to https://data.mendeley.com/datasets/hf6sf8zrkc/2  
> Download the zip, upload to Colab or place it at `data/BanglaLekha-Isolated.zip`

In [ ]:
DATA_DIR = Path('data')
DATA_DIR.mkdir(exist_ok=True)

DATASET_DIR = DATA_DIR / 'BanglaLekha-Isolated'
ZIP_PATH    = DATA_DIR / 'BanglaLekha-Isolated.zip'

# ── Option A: Kaggle API (if you have credentials configured) ────────────────
# !kaggle datasets download -d -p data/  # (no official Kaggle source for this one)

# ── Option B: gdown from a shared Google Drive mirror ───────────────────────
# Replace FILE_ID with the actual ID if you have a Drive mirror
GDRIVE_FILE_ID = None  # e.g. '1AbCdEfGhIjKlMnOpQrStUvWxYz'

if not DATASET_DIR.exists():
    if ZIP_PATH.exists():
        print("📦 Found zip locally, extracting...")
    elif GDRIVE_FILE_ID:
        import gdown
        print("⬇️  Downloading from Google Drive...")
        gdown.download(f'https://drive.google.com/uc?id={GDRIVE_FILE_ID}', str(ZIP_PATH), quiet=False)
    else:
        print("⚠️  Please download the dataset manually from:")
        print("    https://data.mendeley.com/datasets/hf6sf8zrkc/2")
        print(f"   Then place the zip at: {ZIP_PATH.resolve()}")
        raise FileNotFoundError("Dataset not found. See instructions above.")

    print("📂 Extracting...")
    with zipfile.ZipFile(ZIP_PATH, 'r') as zf:
        zf.extractall(DATA_DIR)
    print("✅ Extraction complete!")
else:
    print(f"✅ Dataset already exists at: {DATASET_DIR}")

In [ ]:
# ── Auto-detect the root folder that contains class subdirectories ───────────
def find_dataset_root(base: Path) -> Path:
    """Walk until we find a directory whose children are all class folders."""
    for root, dirs, files in os.walk(base):
        root_p = Path(root)
        img_children = [d for d in dirs if (root_p / d).is_dir()]
        if len(img_children) >= 10:            # at least 10 class folders
            sample = root_p / img_children[0]
            imgs = list(sample.glob('*.[jp][pn]g')) + list(sample.glob('*.bmp'))
            if imgs:
                return root_p
    return base

DATASET_ROOT = find_dataset_root(DATA_DIR)
classes = sorted([d.name for d in DATASET_ROOT.iterdir() if d.is_dir()])
NUM_CLASSES = len(classes)
print(f"📊 Found {NUM_CLASSES} classes at: {DATASET_ROOT}")
print(f"   First 10 classes: {classes[:10]}")

## 3. Dataset Exploration & Visualization

In [ ]:
# Count images per class
class_counts = {}
for cls in classes:
    folder = DATASET_ROOT / cls
    imgs = list(folder.glob('*.[jp][pn]g')) + list(folder.glob('*.bmp')) + list(folder.glob('*.png'))
    class_counts[cls] = len(imgs)

total_images = sum(class_counts.values())
print(f"Total images : {total_images:,}")
print(f"Classes      : {NUM_CLASSES}")
print(f"Avg per class: {total_images / NUM_CLASSES:.0f}")
print(f"Min per class: {min(class_counts.values())}")
print(f"Max per class: {max(class_counts.values())}")

In [ ]:
# Bar chart: images per class
fig, ax = plt.subplots(figsize=(18, 4))
ax.bar(range(NUM_CLASSES), [class_counts[c] for c in classes], color='steelblue', edgecolor='white', linewidth=0.4)
ax.set_xticks(range(NUM_CLASSES))
ax.set_xticklabels(classes, rotation=90, fontsize=7)
ax.set_xlabel('Class')
ax.set_ylabel('Image Count')
ax.set_title('BanglaLekha-Isolated — Images per Class')
plt.tight_layout()
plt.savefig('class_distribution.png', dpi=120)
plt.show()

In [ ]:
# Show sample images from random classes
fig, axes = plt.subplots(4, 10, figsize=(20, 9))
sample_classes = random.sample(classes, min(40, NUM_CLASSES))
for ax, cls in zip(axes.flat, sample_classes):
    folder = DATASET_ROOT / cls
    imgs = list(folder.glob('*.[jp][pn]g')) + list(folder.glob('*.png')) + list(folder.glob('*.bmp'))
    if imgs:
        img = Image.open(random.choice(imgs)).convert('L')
        ax.imshow(img, cmap='gray')
        ax.set_title(cls, fontsize=8)
    ax.axis('off')
plt.suptitle('Sample Bengali Characters (BanglaLekha-Isolated)', fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig('sample_characters.png', dpi=120, bbox_inches='tight')
plt.show()

## 4. Data Loaders with Augmentation

In [ ]:
# ── Hyperparameters ───────────────────────────────────────────────────────────
IMG_SIZE    = 32      # resize all images to 32×32
BATCH_SIZE  = 128
NUM_EPOCHS  = 20
LR_MAX      = 1e-3
WEIGHT_DECAY= 1e-4
VAL_SPLIT   = 0.15
TEST_SPLIT  = 0.10

print(f"Image size  : {IMG_SIZE}×{IMG_SIZE}")
print(f"Batch size  : {BATCH_SIZE}")
print(f"Epochs      : {NUM_EPOCHS}")
print(f"Classes     : {NUM_CLASSES}")

In [ ]:
# ── Transforms ───────────────────────────────────────────────────────────────
train_transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=1),
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomRotation(10),
    transforms.RandomAffine(degrees=0, translate=(0.1, 0.1), scale=(0.9, 1.1)),
    transforms.RandomApply([transforms.GaussianBlur(3)], p=0.2),
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,)),   # → [-1, 1]
])

val_transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=1),
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,)),
])

# ── Load full dataset (with train transform first; we'll swap for val/test) ──
full_dataset = datasets.ImageFolder(root=str(DATASET_ROOT), transform=train_transform)
CLASS_NAMES  = full_dataset.classes
print(f"Classes loaded: {len(CLASS_NAMES)}")
print(f"Total samples : {len(full_dataset):,}")

# ── Stratified-ish split (random_split for simplicity) ───────────────────────
n_total = len(full_dataset)
n_test  = int(n_total * TEST_SPLIT)
n_val   = int(n_total * VAL_SPLIT)
n_train = n_total - n_val - n_test

train_set, val_set, test_set = random_split(
    full_dataset, [n_train, n_val, n_test],
    generator=torch.Generator().manual_seed(SEED)
)

# Override transform for val & test subsets
class TransformSubset(Dataset):
    def __init__(self, subset, transform):
        self.subset    = subset
        self.transform = transform
    def __len__(self):  return len(self.subset)
    def __getitem__(self, idx):
        img, label = self.subset.dataset.samples[self.subset.indices[idx]]
        img = Image.open(img).convert('RGB')
        return self.transform(img), label

val_set  = TransformSubset(val_set,  val_transform)
test_set = TransformSubset(test_set, val_transform)

train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_set,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_set,  batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

print(f"\nSplit  → Train: {n_train:,} | Val: {n_val:,} | Test: {n_test:,}")

In [ ]:
# Visualise one augmented batch
images, labels = next(iter(train_loader))
grid = make_grid(images[:32], nrow=8, normalize=True)
plt.figure(figsize=(14, 5))
plt.imshow(grid.permute(1, 2, 0).cpu(), cmap='gray')
plt.title('Augmented Training Batch (first 32 samples)')
plt.axis('off')
plt.tight_layout()
plt.savefig('augmented_batch.png', dpi=120)
plt.show()
print(f"Batch shape: {images.shape}  Labels shape: {labels.shape}")

## 5. 3-Layer CNN Architecture

```
Input  (1 × 32 × 32)
  └─ Conv Block 1: Conv(32) → BN → ReLU → MaxPool  → 32×16×16
  └─ Conv Block 2: Conv(64) → BN → ReLU → MaxPool  → 64×8×8
  └─ Conv Block 3: Conv(128)→ BN → ReLU → MaxPool  → 128×4×4
  └─ AdaptiveAvgPool → 128×1×1
  └─ FC(256) → Dropout(0.4) → FC(NUM_CLASSES)
```

In [ ]:
class BengaliCNN(nn.Module):
    """3-layer CNN for Bengali character classification (~300k params)."""

    def __init__(self, num_classes: int, dropout: float = 0.4):
        super().__init__()

        # ── Convolutional Blocks ──────────────────────────────────────────────
        self.features = nn.Sequential(
            # Block 1 — 1→32 channels
            nn.Conv2d(1, 32, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.Conv2d(32, 32, kernel_size=3, padding=1, bias=False),  # extra conv for depth
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),      # 32×32 → 16×16
            nn.Dropout2d(0.1),

            # Block 2 — 32→64 channels
            nn.Conv2d(32, 64, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.Conv2d(64, 64, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),      # 16×16 → 8×8
            nn.Dropout2d(0.15),

            # Block 3 — 64→128 channels
            nn.Conv2d(64, 128, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.Conv2d(128, 128, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),      # 8×8 → 4×4
        )

        self.pool = nn.AdaptiveAvgPool2d(1)   # → 128×1×1

        # ── Classifier Head ───────────────────────────────────────────────────
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128, 256),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(256, num_classes),
        )

        # Weight init
        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.ones_(m.weight)
                nn.init.zeros_(m.bias)
            elif isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                nn.init.zeros_(m.bias)

    def forward(self, x):
        x = self.features(x)
        x = self.pool(x)
        return self.classifier(x)


model = BengaliCNN(num_classes=NUM_CLASSES).to(DEVICE)

# Parameter count
total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total parameters    : {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")
print(model)

## 6. Training

In [ ]:
# ── Loss & Optimizer ──────────────────────────────────────────────────────────
criterion  = nn.CrossEntropyLoss(label_smoothing=0.1)
optimizer  = optim.AdamW(model.parameters(), lr=LR_MAX, weight_decay=WEIGHT_DECAY)
scheduler  = optim.lr_scheduler.OneCycleLR(
    optimizer,
    max_lr=LR_MAX,
    steps_per_epoch=len(train_loader),
    epochs=NUM_EPOCHS,
    pct_start=0.3,
    anneal_strategy='cos',
)

print("Criterion : CrossEntropyLoss (label_smoothing=0.1)")
print("Optimizer : AdamW")
print("Scheduler : OneCycleLR")

In [ ]:
def train_one_epoch(model, loader, optimizer, criterion, scheduler, device):
    model.train()
    running_loss, correct, total = 0.0, 0, 0
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss    = criterion(outputs, labels)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        scheduler.step()
        running_loss += loss.item() * images.size(0)
        preds  = outputs.argmax(1)
        correct += (preds == labels).sum().item()
        total  += images.size(0)
    return running_loss / total, correct / total


@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    running_loss, correct, total = 0.0, 0, 0
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        loss    = criterion(outputs, labels)
        running_loss += loss.item() * images.size(0)
        preds  = outputs.argmax(1)
        correct += (preds == labels).sum().item()
        total  += images.size(0)
    return running_loss / total, correct / total

In [ ]:
history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}
best_val_acc  = 0.0
CHECKPOINT    = 'best_bengali_cnn.pt'

print(f"{'Epoch':>6} {'Train Loss':>11} {'Train Acc':>10} {'Val Loss':>10} {'Val Acc':>9} {'LR':>10}")
print('-' * 65)

for epoch in range(1, NUM_EPOCHS + 1):
    train_loss, train_acc = train_one_epoch(model, train_loader, optimizer, criterion, scheduler, DEVICE)
    val_loss,   val_acc   = evaluate(model, val_loader, criterion, DEVICE)

    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)

    current_lr = scheduler.get_last_lr()[0]
    marker = ' ✅' if val_acc > best_val_acc else ''
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save({'epoch': epoch, 'model_state': model.state_dict(),
                    'val_acc': val_acc, 'class_names': CLASS_NAMES}, CHECKPOINT)

    print(f"{epoch:>6} {train_loss:>11.4f} {train_acc*100:>9.2f}% {val_loss:>10.4f} {val_acc*100:>8.2f}%  {current_lr:>9.2e}{marker}")

print(f"\n🏆 Best Validation Accuracy: {best_val_acc*100:.2f}%")

In [ ]:
# ── Training curves ───────────────────────────────────────────────────────────
epochs_range = range(1, NUM_EPOCHS + 1)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(epochs_range, history['train_loss'], 'b-o', label='Train', markersize=4)
ax1.plot(epochs_range, history['val_loss'],   'r-o', label='Val',   markersize=4)
ax1.set_title('Loss'); ax1.set_xlabel('Epoch'); ax1.set_ylabel('Loss')
ax1.legend(); ax1.grid(alpha=0.3)

ax2.plot(epochs_range, [a*100 for a in history['train_acc']], 'b-o', label='Train', markersize=4)
ax2.plot(epochs_range, [a*100 for a in history['val_acc']],   'r-o', label='Val',   markersize=4)
ax2.axhline(97, color='green', linestyle='--', alpha=0.5, label='97% target')
ax2.set_title('Accuracy'); ax2.set_xlabel('Epoch'); ax2.set_ylabel('Accuracy (%)')
ax2.legend(); ax2.grid(alpha=0.3)

plt.suptitle('BengaliCNN Training Curves', fontsize=14)
plt.tight_layout()
plt.savefig('training_curves.png', dpi=120)
plt.show()

## 7. Evaluation on Test Set

In [ ]:
# Load best checkpoint
ckpt = torch.load(CHECKPOINT, map_location=DEVICE)
model.load_state_dict(ckpt['model_state'])
print(f"Loaded best model from epoch {ckpt['epoch']} (val acc {ckpt['val_acc']*100:.2f}%)")

test_loss, test_acc = evaluate(model, test_loader, criterion, DEVICE)
print(f"\n📊 Test Loss    : {test_loss:.4f}")
print(f"📊 Test Accuracy: {test_acc*100:.2f}%")

In [ ]:
# Collect all predictions and true labels for full evaluation
@torch.no_grad()
def get_all_preds(model, loader, device):
    model.eval()
    all_preds, all_labels, all_probs = [], [], []
    for images, labels in tqdm(loader, desc='Predicting'):
        images = images.to(device)
        logits = model(images)
        probs  = F.softmax(logits, dim=1)
        preds  = logits.argmax(1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.numpy())
        all_probs.extend(probs.cpu().numpy())
    return np.array(all_preds), np.array(all_labels), np.array(all_probs)

y_pred, y_true, y_probs = get_all_preds(model, test_loader, DEVICE)
print(f"Predictions gathered for {len(y_true):,} test samples")

In [ ]:
# Detailed classification report
print("\n" + "="*60)
print("CLASSIFICATION REPORT (Top-level)")
print("="*60)
report = classification_report(y_true, y_pred, target_names=CLASS_NAMES, digits=3)
print(report)

In [ ]:
# ── Confusion Matrix ──────────────────────────────────────────────────────────
cm = confusion_matrix(y_true, y_pred)
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)  # row-normalised

fig, ax = plt.subplots(figsize=(22, 20))
sns.heatmap(cm_norm, annot=False, fmt='.2f', cmap='Blues',
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES,
            linewidths=0.3, ax=ax, vmin=0, vmax=1)
ax.set_xlabel('Predicted', fontsize=12)
ax.set_ylabel('True',      fontsize=12)
ax.set_title('Normalised Confusion Matrix — BanglaLekha-Isolated', fontsize=14)
plt.xticks(fontsize=6, rotation=90)
plt.yticks(fontsize=6, rotation=0)
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# ── Per-class accuracy bar chart ──────────────────────────────────────────────
per_class_acc = cm_norm.diagonal()
sorted_idx    = np.argsort(per_class_acc)

fig, ax = plt.subplots(figsize=(18, 5))
colors = ['#d73027' if a < 0.90 else '#fee08b' if a < 0.97 else '#1a9850'
          for a in per_class_acc[sorted_idx]]
ax.bar(range(NUM_CLASSES), per_class_acc[sorted_idx] * 100, color=colors)
ax.axhline(97, color='green', linestyle='--', linewidth=1.5, label='97% target')
ax.set_xticks(range(NUM_CLASSES))
ax.set_xticklabels([CLASS_NAMES[i] for i in sorted_idx], rotation=90, fontsize=6)
ax.set_ylabel('Accuracy (%)')
ax.set_title('Per-Class Accuracy (sorted ascending)')
patches = [mpatches.Patch(color='#1a9850', label='≥97%'),
           mpatches.Patch(color='#fee08b', label='90–97%'),
           mpatches.Patch(color='#d73027', label='<90%')]
ax.legend(handles=patches, loc='upper left')
plt.tight_layout()
plt.savefig('per_class_accuracy.png', dpi=120)
plt.show()

print(f"Classes ≥97% accuracy: {(per_class_acc >= 0.97).sum()} / {NUM_CLASSES}")
print(f"Overall test accuracy: {per_class_acc.mean()*100:.2f}%")

In [ ]:
# ── Top-5 Accuracy ────────────────────────────────────────────────────────────
top5_correct = 0
for i, true_label in enumerate(y_true):
    top5 = np.argsort(y_probs[i])[::-1][:5]
    if true_label in top5:
        top5_correct += 1
print(f"Top-5 Accuracy: {top5_correct / len(y_true) * 100:.2f}%")

In [ ]:
# ── Visualise misclassified examples ─────────────────────────────────────────
misclassified_idx = np.where(y_pred != y_true)[0]
print(f"Total misclassified: {len(misclassified_idx)} / {len(y_true)}")

show_n = min(20, len(misclassified_idx))
sample_idx = random.sample(list(misclassified_idx), show_n)

fig, axes = plt.subplots(4, 5, figsize=(14, 11))
for ax, idx in zip(axes.flat, sample_idx):
    # Re-fetch from test_set
    img_tensor, _ = test_set[idx]
    img_np = img_tensor.squeeze().numpy()
    img_np = (img_np * 0.5 + 0.5)  # un-normalise
    ax.imshow(img_np, cmap='gray')
    true_cls  = CLASS_NAMES[y_true[idx]]
    pred_cls  = CLASS_NAMES[y_pred[idx]]
    conf      = y_probs[idx][y_pred[idx]] * 100
    ax.set_title(f"T:{true_cls}\nP:{pred_cls} ({conf:.0f}%)", fontsize=8,
                 color='red')
    ax.axis('off')

plt.suptitle('Misclassified Examples', fontsize=14)
plt.tight_layout()
plt.savefig('misclassified.png', dpi=120, bbox_inches='tight')
plt.show()

## 8. Save Model for Streamlit Deployment

In [ ]:
import json

# Save full model info
export = {
    'model_state': model.state_dict(),
    'class_names': CLASS_NAMES,
    'num_classes': NUM_CLASSES,
    'img_size':    IMG_SIZE,
    'test_acc':    float(test_acc),
}
torch.save(export, 'bengali_cnn_final.pt')
print("✅ Saved: bengali_cnn_final.pt")

# Save class names as JSON (easy to load in Streamlit)
with open('class_names.json', 'w', encoding='utf-8') as f:
    json.dump({'class_names': CLASS_NAMES, 'img_size': IMG_SIZE}, f, ensure_ascii=False, indent=2)
print("✅ Saved: class_names.json")

In [ ]:
# ── Quick inference test ──────────────────────────────────────────────────────
model.eval()
sample_img, sample_label = test_set[0]
with torch.no_grad():
    logits = model(sample_img.unsqueeze(0).to(DEVICE))
    probs  = F.softmax(logits, dim=1)[0]
    top5   = probs.topk(5)

print(f"True class: {CLASS_NAMES[sample_label]}")
print("Top-5 predictions:")
for i, (score, idx) in enumerate(zip(top5.values, top5.indices)):
    print(f"  {i+1}. {CLASS_NAMES[idx.item()]:20s}  {score.item()*100:.1f}%")

## 9. Streamlit App Code

Save the cell below as `app.py` and run: `streamlit run app.py`

**Requirements:** `pip install streamlit streamlit-drawable-canvas torch torchvision pillow`

In [ ]:
streamlit_app_code = '''
# app.py — Bengali Character Recognition with Drawable Canvas
# Run: streamlit run app.py

import json, random
import numpy as np
import streamlit as st
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import transforms
from PIL import Image
from streamlit_drawable_canvas import st_canvas


# ── Model definition (must match training) ────────────────────────────────────
class BengaliCNN(nn.Module):
    def __init__(self, num_classes, dropout=0.4):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 32, 3, padding=1, bias=False), nn.BatchNorm2d(32), nn.ReLU(True),
            nn.Conv2d(32, 32, 3, padding=1, bias=False), nn.BatchNorm2d(32), nn.ReLU(True),
            nn.MaxPool2d(2, 2), nn.Dropout2d(0.1),
            nn.Conv2d(32, 64, 3, padding=1, bias=False), nn.BatchNorm2d(64), nn.ReLU(True),
            nn.Conv2d(64, 64, 3, padding=1, bias=False), nn.BatchNorm2d(64), nn.ReLU(True),
            nn.MaxPool2d(2, 2), nn.Dropout2d(0.15),
            nn.Conv2d(64, 128, 3, padding=1, bias=False), nn.BatchNorm2d(128), nn.ReLU(True),
            nn.Conv2d(128, 128, 3, padding=1, bias=False), nn.BatchNorm2d(128), nn.ReLU(True),
            nn.MaxPool2d(2, 2),
        )
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.classifier = nn.Sequential(
            nn.Flatten(), nn.Linear(128, 256), nn.ReLU(True), nn.Dropout(dropout),
            nn.Linear(256, num_classes),
        )
    def forward(self, x):
        return self.classifier(self.pool(self.features(x)))


@st.cache_resource
def load_model():
    with open("class_names.json") as f:
        info = json.load(f)
    class_names = info["class_names"]
    img_size    = info["img_size"]
    ckpt = torch.load("bengali_cnn_final.pt", map_location="cpu")
    model = BengaliCNN(num_classes=len(class_names))
    model.load_state_dict(ckpt["model_state"])
    model.eval()
    return model, class_names, img_size


def preprocess(canvas_data, img_size):
    img = Image.fromarray(canvas_data.astype(np.uint8)).convert("L")
    tf  = transforms.Compose([
        transforms.Resize((img_size, img_size)),
        transforms.ToTensor(),
        transforms.Normalize((0.5,), (0.5,)),
    ])
    return tf(img).unsqueeze(0)


# ── UI ────────────────────────────────────────────────────────────────────────
st.set_page_config(page_title="বাংলা অক্ষর পরিচিতি", page_icon="✍️", layout="centered")
st.title("✍️ Bengali Character Recognition")
st.caption("Draw a Bengali character → get live Unicode prediction (BanglaLekha-Isolated CNN)")

model, class_names, img_size = load_model()
mode = st.radio("Mode", ["Free Draw", "Practice Mode"], horizontal=True)

if mode == "Practice Mode":
    if "target" not in st.session_state:
        st.session_state.target = random.choice(class_names)
    st.info(f"✏️ Practice drawing: **{st.session_state.target}**")
    if st.button("New character"):
        st.session_state.target = random.choice(class_names)
        st.experimental_rerun()

col1, col2 = st.columns([1.2, 1])
with col1:
    canvas_result = st_canvas(
        fill_color="rgba(0,0,0,0)",
        stroke_width=st.slider("Stroke width", 5, 25, 14),
        stroke_color="#000000",
        background_color="#FFFFFF",
        height=280, width=280,
        drawing_mode="freedraw",
        key="canvas",
    )

with col2:
    if canvas_result.image_data is not None:
        pixel_sum = canvas_result.image_data[:, :, :3].sum()
        if pixel_sum < 280 * 280 * 3 * 255 * 0.999:  # something drawn
            inp = preprocess(canvas_result.image_data[:, :, :3], img_size)
            with torch.no_grad():
                logits = model(inp)
                probs  = F.softmax(logits, dim=1)[0]
                top5   = probs.topk(5)

            st.subheader("Prediction")
            best_cls  = class_names[top5.indices[0]]
            best_conf = top5.values[0].item() * 100
            st.metric("Top-1", best_cls, f"{best_conf:.1f}% confidence")

            st.subheader("Top-5")
            for score, idx in zip(top5.values, top5.indices):
                cls  = class_names[idx.item()]
                conf = score.item() * 100
                st.progress(int(conf), text=f"{cls}  —  {conf:.1f}%")

            if mode == "Practice Mode":
                target = st.session_state.target
                if best_cls == target:
                    st.success(f"🎉 Correct! You drew '{target}' perfectly!")
                else:
                    st.warning(f"Try again! You drew '{best_cls}', target is '{target}'.")
        else:
            st.info("☝️ Draw a character on the canvas")

st.sidebar.markdown("### About")
st.sidebar.write(f"**Model:** 3-layer CNN  |  **Classes:** {len(class_names)}")
st.sidebar.write("**Dataset:** BanglaLekha-Isolated (~166k images)")
'''

with open('app.py', 'w', encoding='utf-8') as f:
    f.write(streamlit_app_code.strip())
print("✅ Streamlit app saved as app.py")
print("   Run with: streamlit run app.py")

## 10. Summary

| Metric | Value |
|---|---|
| Dataset | BanglaLekha-Isolated |
| Classes | 84 Bengali characters |
| Total images | ~166,000 |
| Model | 3-layer CNN (BengaliCNN) |
| Parameters | ~300k |
| Image size | 32×32 (grayscale) |
| Best Val Accuracy | see training output |
| Test Accuracy | see evaluation output |
| Training time | < 5 min on Colab T4 |

### Files Generated
- `best_bengali_cnn.pt` — best checkpoint during training  
- `bengali_cnn_final.pt` — final export with class names  
- `class_names.json` — class list for Streamlit  
- `app.py` — Streamlit drawable-canvas app  
- `training_curves.png`, `confusion_matrix.png`, `per_class_accuracy.png`